# Notebook 20: AquaBoundaryLoss & water-bodies-detection Capstone

**Module:** 07 Segmentation  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Walk through complete loss from losses.py
2. Understand training metrics
3. Preview full pipeline
4. Complete Module 07

## Your Complete Loss (losses.py)

```python
class AquaBoundaryLoss(nn.Module):
    def forward(self, logits, targets):
        la = BCEDiceLoss()(logits[:, 0:1], targets[:, 0:1])  # aqua
        lb = BCEDiceLoss()(logits[:, 1:2], targets[:, 1:2])  # boundary
        return w_aqua * la + w_boundary * lb
```

**BCEDiceLoss:** 0.35 × BCE + 0.65 × Dice per channel

**Why BCE + Dice?** BCE for pixel-wise accuracy, Dice for region overlap (handles imbalance).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Full reimplementation matching your losses.py
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.35, dice_weight=0.65):
        super().__init__()
        self.bw, self.dw = bce_weight, dice_weight
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float())
        dice = 1 - dice_with_logits(logits, targets)
        return self.bw * bce + self.dw * dice

class AquaBoundaryLoss(nn.Module):
    def __init__(self, w_aqua=1.0, w_boundary=0.35):
        super().__init__()
        self.core = BCEDiceLoss()
        self.w_aqua, self.w_boundary = w_aqua, w_boundary
    def forward(self, logits, targets):
        la = self.core(logits[:, 0:1], targets[:, 0:1])
        lb = self.core(logits[:, 1:2], targets[:, 1:2])
        return self.w_aqua * la + self.w_boundary * lb

loss_fn = AquaBoundaryLoss()
logits = torch.randn(2, 2, 64, 64)
targets = torch.zeros(2, 2, 64, 64)
targets[:, 0, 10:50, 10:50] = 1.0
targets[:, 1, 9:51, 9:51] = 1.0
print(f'AquaBoundaryLoss: {loss_fn(logits, targets):.4f}')

## water-bodies-detection Pipeline Preview

```
1. tile_and_mask.py   → Planet GeoTIFF + shapefile → tiles + dual masks
2. dataset.py         → Albumentations augmentation
3. model.py           → UNet++ SE-ResNet50 (Module 06+08)
4. losses.py          → AquaBoundaryLoss (this notebook)
5. train.py           → Two-stage training, early stopping on val IoU
6. predict.py         → Sliding window + TTA + Hann blending
7. post_process/      → Threshold → polygons → shapefile
```

**Full walkthrough:** Module 12 Capstone

## Module 07 Metrics Summary

| Metric | Formula | Your project |
|--------|---------|-------------|
| IoU | intersection/union | Early stopping metric |
| Dice | 2×inter/(sum) | Part of loss |
| Pixel Acc | correct/total | Monitoring |
| mIoU | mean IoU classes | Multi-class extension |

## Exercise

Train UNet from Notebook 07 on synthetic data with BCEDiceLoss. Plot IoU vs epoch.

In [ ]:
# YOUR CODE HERE


## Module 07 Complete

**Next:** Module 08 Object Detection — R-CNN, YOLO, DETR.

---

## Summary

AquaBoundaryLoss combines BCE+Dice on dual heads — the heart of your production pipeline.